# Benchmarking `bcpd` on the Turing Change Point Dataset

This notebook runs the offline and online algorithms across a corpus of real
annotated time series and scores them against human annotations.

It is also the fastest way to learn the API: every cell below is a small,
complete example you can copy.

**What it does**

1. Loads the Turing Change Point Dataset (TCPD) — 37 real series, each annotated
   by several people — or falls back to synthetic series if TCPD is unavailable.
2. Walks through a single series in detail, offline and online.
3. Scores every series with the two metrics used in the TCPD paper.
4. Compares against a reference Adams & MacKay BOCPD implementation.

**Before running**

    pip install -e ".[dev,notebook]"

`plotly` draws the figures; `nbformat` is what lets plotly render inside Jupyter.
Both come with the `notebook` extra. If a figure raises *"Mime type rendering
requires nbformat"*, install it and **restart the kernel** — plotly checks for it
once, at import, so a mid-session install has no effect.

The setup cell below locates the project by walking up from wherever the kernel
started, then puts `src/` and `benchmarks/` on the path. That means the notebook
runs whether or not `pip install -e .` succeeded, and whether Jupyter was opened
from `notebooks/`, the repo root, or anywhere else.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

# Set this only if the automatic search below fails — it should be the folder
# containing pyproject.toml, src/ and benchmarks/.
# e.g. PROJECT_ROOT = r"C:\Users\nasta\Documents\GIT\bcpd-files\bcpd-python-port\bcpd"
PROJECT_ROOT = None


def find_project_root(start=None):
    """Walk up from `start` looking for the folder that holds the package."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "bcpd").is_dir():
            return candidate
    return None


ROOT = Path(PROJECT_ROOT).resolve() if PROJECT_ROOT else find_project_root()
if ROOT is None:
    raise RuntimeError(
        f"Could not find the project root from {Path.cwd()}.\n"
        "Set PROJECT_ROOT above to the folder containing pyproject.toml and src/."
    )

# src/ makes `import bcpd` work even if `pip install -e .` did not take;
# benchmarks/ makes compare.py importable for the BOCPD comparison in section 7.
for extra in (ROOT / "src", ROOT / "benchmarks"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import json
import time
import warnings

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import bcpd
from bcpd import (
    OnlineChangepointFilter,
    PriorSpec,
    SegmentLengthPrior,
    SegmentLikelihood,
    constant_design,
    detect,
    trend_design,
)

warnings.filterwarnings("ignore")
PALETTE = dict(data="#4C6EF5", truth="#E8590C", found="#0CA678", band="rgba(76,110,245,0.18)")

print(f"project root : {ROOT}")
print(f"bcpd         : {Path(bcpd.__file__).parent}")
print(f"python       : {sys.executable}")

## 2. The data

TCPD is distributed as one JSON file per series. The loader below tries, in order:

1. a local copy, if you set `TCPD_PATH` to a cloned checkout;
2. a download from GitHub;
3. synthetic series with known changepoints, so the rest of the notebook still runs.

To use the real corpus:

    git clone https://github.com/alan-turing-institute/TCPD.git
    cd TCPD && python build_tcpd.py

Some series are fetched from their original sources at build time for licensing
reasons, which is why a plain clone is not always enough.

In [ ]:
TCPD_PATH = None   # e.g. Path.home() / "TCPD"

TCPD_RAW = "https://raw.githubusercontent.com/alan-turing-institute/TCPD/master/datasets"
ANNOTATIONS_URL = (
    "https://raw.githubusercontent.com/alan-turing-institute/TCPDBench/master/"
    "analysis/annotations/annotations.json"
)


def _parse(obj):
    """TCPD JSON -> (values, name). Multivariate series become (n, d) arrays."""
    cols = [np.asarray(s["raw"], dtype=float) for s in obj["series"]]
    y = np.column_stack(cols) if len(cols) > 1 else cols[0]
    return y, obj.get("name", "unnamed")


def load_local(path):
    out = {}
    for f in sorted(Path(path).rglob("*.json")):
        if f.name in {"annotations.json", "schema.json"}:
            continue
        try:
            obj = json.loads(f.read_text())
            if "series" in obj:
                y, name = _parse(obj)
                out[name] = y
        except Exception:
            pass
    return out


def load_remote(names):
    import urllib.request
    out = {}
    for n in names:
        try:
            with urllib.request.urlopen(f"{TCPD_RAW}/{n}/{n}.json", timeout=20) as r:
                y, name = _parse(json.loads(r.read().decode()))
                out[name] = y
        except Exception:
            pass
    return out


def load_annotations():
    import urllib.request
    try:
        with urllib.request.urlopen(ANNOTATIONS_URL, timeout=20) as r:
            return json.loads(r.read().decode())
    except Exception:
        return {}


def synthetic_corpus(n_series=12, seed=0):
    """Fallback: piecewise-constant series with known changepoints."""
    rng = np.random.default_rng(seed)
    data, truth = {}, {}
    for i in range(n_series):
        n = int(rng.integers(300, 700))
        k = int(rng.integers(1, 4))
        cuts = sorted(rng.choice(np.arange(n // 6, n - n // 6), size=k, replace=False))
        y = rng.normal(0, 1, n)
        for c in cuts:
            y[c:] += rng.normal(0, 1) * rng.choice([-1, 1]) * rng.uniform(2.5, 6)
        name = f"synthetic_{i:02d}"
        data[name] = y
        truth[name] = {"synthetic": [int(c) for c in cuts]}
    return data, truth

In [ ]:
TCPD_NAMES = [
    "apple", "bank", "bee_waggle_6", "bitcoin", "brent_spot", "businv",
    "centralia", "children_per_woman", "co2_canada", "construction",
    "debt_ireland", "gdp_argentina", "gdp_croatia", "gdp_iran", "gdp_japan",
    "global_co2", "homeruns", "iceland_tourism", "jfk_passengers",
    "lga_passengers", "measles", "nile", "occupancy", "ozone", "quality_control_1",
    "quality_control_2", "quality_control_3", "quality_control_4",
    "quality_control_5", "rail_lines", "ratner_stock", "robocalls",
    "scanline_126007", "scanline_42049", "seatbelts", "shanghai_license",
    "uk_coal_employ", "unemployment_nl", "usd_isk", "us_population", "well_log",
]

series, annotations = {}, {}
if TCPD_PATH is not None and Path(TCPD_PATH).exists():
    series = load_local(TCPD_PATH)
    print(f"loaded {len(series)} series from {TCPD_PATH}")
if not series:
    series = load_remote(TCPD_NAMES)
    print(f"downloaded {len(series)} series")
if series:
    annotations = load_annotations()
    print(f"annotations for {len(annotations)} series")
if not series:
    print("TCPD unavailable, using synthetic corpus")
    series, annotations = synthetic_corpus()

print(f"\n{len(series)} series ready")
for name in list(series)[:5]:
    y = series[name]
    print(f"  {name:22s} n={len(y):5d}  d={1 if y.ndim == 1 else y.shape[1]}")

## 3. One series in detail

Start with a single series so the output of each algorithm is visible before
running everything at once.

`detect` needs a design matrix and a `min_distance`. The design decides what
counts as a change: `constant_design` detects shifts in level and is the right
default; `trend_design` adds a slope, which lets a segment absorb a moderate
level shift as drift, so a step that `constant_design` flags clearly may not be
flagged at all.

In [ ]:
name = "well_log" if "well_log" in series else list(series)[0]
y = series[name]
if y.ndim > 1:
    y = y[:, 0]
y = (y - np.nanmean(y)) / np.nanstd(y)      # scale-free, so one prior fits all series
y = np.nan_to_num(y, nan=float(np.nanmedian(y)))
n = len(y)

min_distance = max(10, n // 25)

t0 = time.perf_counter()
offline = detect(
    y, constant_design(n),
    min_distance=min_distance,
    k_max=8,
    n_samples=300,
    rng=np.random.default_rng(0),
)
elapsed = time.perf_counter() - t0

k = offline.posterior.map_count()
print(f"{name}: n={n}, min_distance={min_distance}")
print(f"  {k} changepoints (posterior {offline.n_changepoints[k]:.3f}) in {elapsed:.2f}s")
print(f"  full posterior over count: {np.round(offline.n_changepoints, 3)}")

In [ ]:
def peaks(prob, k, min_sep):
    """The k most probable locations, kept at least min_sep apart."""
    p, out = prob.copy(), []
    for _ in range(k):
        i = int(np.argmax(p))
        if p[i] <= 0:
            break
        out.append(i)
        p[max(0, i - min_sep): i + min_sep] = 0.0
    return sorted(out)


def plot_series(y, prob, found, truth=None, title=""):
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, row_heights=[0.68, 0.32],
        vertical_spacing=0.06,
        subplot_titles=(title, "posterior probability of a changepoint"),
    )
    fig.add_trace(go.Scatter(y=y, mode="lines", name="series",
                             line=dict(color=PALETTE["data"], width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(y=prob, mode="lines", name="P(changepoint)",
                             line=dict(color=PALETTE["found"], width=1.2),
                             fill="tozeroy", fillcolor=PALETTE["band"]), row=2, col=1)
    for c in found:
        fig.add_vline(x=c, line=dict(color=PALETTE["found"], width=1.5, dash="solid"),
                      row=1, col=1)
    for c in (truth or []):
        fig.add_vline(x=c, line=dict(color=PALETTE["truth"], width=1.5, dash="dot"),
                      row=1, col=1)
    fig.update_layout(height=560, template="plotly_white",
                      legend=dict(orientation="h", y=1.08),
                      margin=dict(l=50, r=30, t=70, b=40))
    return fig


truth = []
if name in annotations:
    all_ann = [c for v in annotations[name].values() for c in v]
    truth = sorted(set(all_ann))

found = peaks(offline.location_prob, k, min_distance)
plot_series(y, offline.location_prob, found, truth,
            f"{name} — solid green: detected, dotted orange: annotated").show()

### The same series online

The offline algorithm sees everything at once and returns a posterior over the
whole set of changepoints. The online filter processes one observation at a time
and maintains a posterior over the *run length* — how long the current regime has
lasted. A collapse in run length is the filter announcing a change.

Expect a step or two of detection lag: one observation from a new regime is not
yet enough evidence to outweigh "the current segment continues".

In [ ]:
likelihood = SegmentLikelihood(y, constant_design(n), PriorSpec())
filt = OnlineChangepointFilter(
    likelihood,
    SegmentLengthPrior(mean_length=n / 6.0, min_length=min_distance),
    max_particles=150,
)
res = filt.run(rng=np.random.default_rng(0))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45],
                    vertical_spacing=0.06,
                    subplot_titles=(f"{name}", "most probable run length"))
fig.add_trace(go.Scatter(y=y, mode="lines", line=dict(color=PALETTE["data"], width=1),
                         name="series"), row=1, col=1)
fig.add_trace(go.Scatter(y=res.map_run_length, mode="lines",
                         line=dict(color=PALETTE["found"], width=1.2),
                         name="run length"), row=2, col=1)
for c in truth:
    fig.add_vline(x=c, line=dict(color=PALETTE["truth"], width=1.2, dash="dot"))
fig.update_layout(height=560, template="plotly_white",
                  legend=dict(orientation="h", y=1.08),
                  margin=dict(l=50, r=30, t=70, b=40))
fig.show()

draws = filt.backward_sample(300, rng=np.random.default_rng(1))
counts = np.bincount([len(d) for d in draws], minlength=10)
print("online changepoints per draw:",
      {i: int(c) for i, c in enumerate(counts) if c})

## 4. Scoring

Two metrics from the TCPD paper, both defined against multiple annotators.

**Covering** treats a segmentation as a partition and measures overlap with the
annotator's partition, weighted by segment length. It rewards getting segment
*boundaries* roughly right and does not require exact hits.

**F1 with a margin** counts a detection as correct if it lands within a margin of
an annotated changepoint, then takes the usual harmonic mean of precision and
recall.

Each series is scored against every annotator and the best score is kept, which
is how the original benchmark handles disagreement between annotators.

In [ ]:
def to_partition(locations, n):
    bounds = [0, *sorted(locations), n]
    return [set(range(bounds[i], bounds[i + 1])) for i in range(len(bounds) - 1)
            if bounds[i + 1] > bounds[i]]


def covering(found, true_cps, n):
    """Weighted Jaccard overlap between the two partitions."""
    G, S = to_partition(true_cps, n), to_partition(found, n)
    total = 0.0
    for A in G:
        best = max((len(A & B) / len(A | B) for B in S), default=0.0)
        total += len(A) * best
    return total / n


def f1_margin(found, true_cps, margin):
    if not true_cps:
        return 1.0 if not found else 0.0
    if not found:
        return 0.0
    tp = sum(any(abs(f - t) <= margin for f in found) for t in true_cps)
    precision = sum(any(abs(f - t) <= margin for t in true_cps) for f in found) / len(found)
    recall = tp / len(true_cps)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def score(found, ann_for_series, n, margin):
    """Best score across annotators, as in the TCPD benchmark."""
    if not ann_for_series:
        return None, None
    covers = [covering(found, list(v), n) for v in ann_for_series.values()]
    f1s = [f1_margin(found, list(v), margin) for v in ann_for_series.values()]
    return max(covers), max(f1s)


print(f"sanity check — perfect detection covering: {covering([100, 200], [100, 200], 300):.3f}")
print(f"              no detection covering:      {covering([], [100, 200], 300):.3f}")
print(f"              perfect F1:                 {f1_margin([100, 200], [100, 200], 5):.3f}")

## 5. Every series

In [ ]:
def prepare(y):
    if y.ndim > 1:
        y = y[:, 0]
    y = np.asarray(y, dtype=float)
    y = np.nan_to_num(y, nan=float(np.nanmedian(y)))
    sd = y.std()
    return (y - y.mean()) / (sd if sd > 0 else 1.0)


results = []
for name, raw in series.items():
    y = prepare(raw)
    n = len(y)
    if n < 40:
        continue
    md_ = max(10, n // 25)
    if 2 * md_ > n:
        continue

    t0 = time.perf_counter()
    r = detect(y, constant_design(n), min_distance=md_, k_max=8,
               n_samples=200, rng=np.random.default_rng(0))
    dt = time.perf_counter() - t0

    k = r.posterior.map_count()
    found = peaks(r.location_prob, k, md_)
    cov, f1 = score(found, annotations.get(name, {}), n, margin=max(5, n // 100))
    results.append(dict(name=name, n=n, k=k, found=found,
                        covering=cov, f1=f1, seconds=dt,
                        prob=r.location_prob, series=y))

print(f"{len(results)} series scored\n")
print(f"{'series':24s} {'n':>6} {'k':>3} {'cover':>7} {'F1':>6} {'sec':>7}")
print("-" * 58)
for r in results:
    c = f"{r['covering']:.3f}" if r["covering"] is not None else "   -  "
    f = f"{r['f1']:.3f}" if r["f1"] is not None else "  -  "
    print(f"{r['name'][:24]:24s} {r['n']:6d} {r['k']:3d} {c:>7} {f:>6} {r['seconds']:7.2f}")

scored = [r for r in results if r["covering"] is not None]
if scored:
    print(f"\nmean covering {np.mean([r['covering'] for r in scored]):.3f}"
          f"   mean F1 {np.mean([r['f1'] for r in scored]):.3f}")
print(f"total runtime {sum(r['seconds'] for r in results):.1f}s")

## 6. Results at a glance

In [ ]:
scored = [r for r in results if r["covering"] is not None] or results
order = sorted(scored, key=lambda r: (r["covering"] or 0))

fig = make_subplots(rows=1, cols=2, subplot_titles=("covering", "F1"),
                    horizontal_spacing=0.14)
fig.add_trace(go.Bar(y=[r["name"] for r in order],
                     x=[r["covering"] or 0 for r in order],
                     orientation="h", marker_color=PALETTE["found"],
                     showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(y=[r["name"] for r in order],
                     x=[r["f1"] or 0 for r in order],
                     orientation="h", marker_color=PALETTE["data"],
                     showlegend=False), row=1, col=2)
fig.update_layout(height=max(420, 22 * len(order)), template="plotly_white",
                  margin=dict(l=160, r=30, t=60, b=40))
fig.update_xaxes(range=[0, 1])
fig.show()

In [ ]:
# Runtime against series length: the offline algorithm is O(N^2).
fig = go.Figure()
fig.add_trace(go.Scatter(x=[r["n"] for r in results], y=[r["seconds"] for r in results],
                         mode="markers+text", text=[r["name"] for r in results],
                         textposition="top center", textfont=dict(size=8),
                         marker=dict(size=9, color=PALETTE["data"]), name="series"))
ns = np.array(sorted(r["n"] for r in results), dtype=float)
if len(ns) > 1:
    ref = (ns / ns[-1]) ** 2 * max(r["seconds"] for r in results)
    fig.add_trace(go.Scatter(x=ns, y=ref, mode="lines", name="O(N²) reference",
                             line=dict(color=PALETTE["truth"], dash="dash", width=1.5)))
fig.update_layout(template="plotly_white", height=460,
                  xaxis_title="series length", yaxis_title="seconds",
                  margin=dict(l=60, r=30, t=50, b=50))
fig.show()

### The worst results are worth looking at

A low covering score is usually diagnostic rather than a failure. Common causes:

- the series trends, so `constant_design` chops the trend into levels — try
  `trend_design`;
- the series is periodic, so the cycle is segmented — add Fourier terms with
  `joint_design` and check `posterior_size`;
- `min_distance` is wrong for the sampling rate;
- the annotators disagreed with each other, which the corpus does not hide.

In [ ]:
worst = sorted(scored, key=lambda r: (r["covering"] or 0))[:3]
for r in worst:
    truth_r = sorted({c for v in annotations.get(r["name"], {}).values() for c in v})
    plot_series(r["series"], r["prob"], r["found"], truth_r,
                f"{r['name']} — covering {r['covering']:.3f}").show()

## 7. Against BOCPD

The reference Adams & MacKay implementation from `benchmarks/compare.py`. It
shares the online filter's complexity but updates sufficient statistics in O(1)
per run length with array operations, so it is much faster in wall-clock while
handling only a change in mean and variance.

In [ ]:
# benchmarks/ went on sys.path in the setup cell, so this needs no path fiddling.
try:
    from compare import bocpd_changepoints, bocpd_normal
except ImportError as exc:
    print(f"could not import compare.py: {exc}")
    print(f"expected it at: {ROOT / 'benchmarks' / 'compare.py'}")
else:
    rows = []
    for r in results[:12]:
        y_r, n_r = r["series"], r["n"]
        t0 = time.perf_counter()
        R = bocpd_normal(y_r, hazard=1.0 / max(20, n_r // 10))
        dt = time.perf_counter() - t0
        found_r = bocpd_changepoints(R)
        cov_r, _ = score(found_r, annotations.get(r["name"], {}), n_r, max(5, n_r // 100))
        rows.append((r["name"], r["covering"], cov_r, r["seconds"], dt))

    def fmt(v):
        return f"{v:.3f}" if v is not None else "  -  "

    print(f"{'series':24s} {'bcpd cov':>9} {'BOCPD cov':>10} {'bcpd s':>8} {'BOCPD s':>8}")
    print("-" * 62)
    for name_, c_bcpd, c_bocpd, s_bcpd, s_bocpd in rows:
        print(f"{name_[:24]:24s} {fmt(c_bcpd):>9} {fmt(c_bocpd):>10} "
              f"{s_bcpd:8.2f} {s_bocpd:8.2f}")

    print("\nBOCPD is far faster in wall-clock and handles only a change in mean")
    print("and variance. bcpd takes any design matrix, and is the only one here")
    print("that reports how many changepoints there are.")

## Notes

**Everything here uses one setting for every series.** `min_distance = n/25`,
`k_max = 8`, `constant_design`, standardised input. That is deliberate — it is
the honest way to report a benchmark, since tuning per series would flatter the
method. Real use on a single series should set these from what you know about it.

**Standardising the input matters.** Dividing by the standard deviation makes one
prior reasonable across series spanning wildly different scales. Without it,
`PriorSpec` defaults would be badly wrong on most of the corpus.

**The annotators disagree**, which is why each series is scored against the best
matching annotation rather than a single ground truth. Series where annotators
disagree strongly are not really solvable, and the covering scores reflect that.